[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/063.%20The%20Zone%20Picking%20%26%20Pick-and-Pass%20Balancing%20Problem/P63-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 63. The Zone Picking & Pick-and-Pass Balancing Problem
## Tier 1: Multi-Objective Mathematical Formulation

### Problem Context

Zone picking represents one of the most sophisticated warehouse fulfillment strategies, where the storage area is divided into distinct zones with specialized pickers. The effectiveness of this system hinges critically on achieving optimal balance across zones and seamless coordination in pick-and-pass operations. When orders flow through multiple zones sequentially, any imbalance in workload distribution can create devastating bottlenecks.

### Key Assumptions
- Orders flow sequentially through zones via automated conveyor belts
- Each zone has specific capacity constraints and processing times
- Handoff times between zones are constant and predictable
- Worker capacity is limited per time period
- Multiple objectives must be optimized simultaneously

### Approach (Step-by-Step)

1. **Multi-Objective Formulation**: We'll formulate the problem as a multi-objective mixed-integer programming model that simultaneously optimizes:
   - Total completion time (makespan)
   - Workload balance across zones
   - Total flow time
   - System throughput

2. **Mathematical Model**: Define sets, parameters, decision variables, objective functions, and constraints

3. **Solution Method**: Use weighted sum approach to convert multi-objective to single-objective

4. **Analysis**: Perform sensitivity analysis and visualize results

### What to Look for in the Results
- Pareto-optimal solutions balancing multiple objectives
- Trade-offs between completion time and workload balance
- Zone utilization patterns
- Order assignment schedules

### Concrete Example

We'll implement the three-zone system from the source:
- 3 zones with capacities [100, 80, 60] items/hour
- 2 workers per zone
- 4 orders with varying item requirements per zone
- Pick times: Zone 1 = 2 min/item, Zone 2 = 1.5 min/item, Zone 3 = 3 min/item
- Handoff time = 2 minutes between zones

In [1]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import pulp
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Zone:
    """Represents a picking zone in the warehouse"""
    zone_id: int
    capacity: float  # items per hour
    pick_time: float  # minutes per item
    workers: int
    
@dataclass
class Order:
    """Represents a customer order"""
    order_id: int
    items_by_zone: Dict[int, int]  # zone_id -> item_count
    
@dataclass
class ZoneAssignment:
    """Stores assignment decisions"""
    order_zone_assignments: Dict[int, Dict[int, Tuple[int, int]]]  # order_id -> zone_id -> (start_time, end_time)
    zone_utilizations: Dict[int, float]
    completion_times: Dict[int, int]

# Define the concrete example from the source
zones = [
    Zone(1, 100, 2.0, 2),  # Zone 1: 100 items/hour, 2 min/item, 2 workers
    Zone(2, 80, 1.5, 2),   # Zone 2: 80 items/hour, 1.5 min/item, 2 workers
    Zone(3, 60, 3.0, 2)    # Zone 3: 60 items/hour, 3 min/item, 2 workers
]

orders = [
    Order(1, {1: 5, 2: 3, 3: 2}),  # Order 1: 5 items Zone 1, 3 items Zone 2, 2 items Zone 3
    Order(2, {1: 2, 3: 4}),        # Order 2: 2 items Zone 1, 4 items Zone 3
    Order(3, {2: 3, 3: 1}),        # Order 3: 3 items Zone 2, 1 item Zone 3
    Order(4, {1: 4, 2: 2})         # Order 4: 4 items Zone 1, 2 items Zone 2
]

handoff_time = 2  # minutes between zones

print(f"Zone Configuration:")
for zone in zones:
    print(f"  Zone {zone.zone_id}: {zone.capacity} items/hour, {zone.pick_time} min/item, {zone.workers} workers")

print(f"\nOrder Requirements:")
for order in orders:
    print(f"  Order {order.order_id}: {order.items_by_zone}")

In [ ]:
class MultiObjectiveZoneOptimizer:
    """Multi-objective optimization for zone picking and pick-and-pass balancing"""
    
    def __init__(self, zones: List[Zone], orders: List[Order], handoff_time: float):
        self.zones = zones
        self.orders = orders
        self.handoff_time = handoff_time
        self.time_periods = 20  # Planning horizon in time periods
        
    def solve_multi_objective(self, weights: Dict[str, float] = None) -> Dict:
        """Solve multi-objective optimization using weighted sum approach"""
        if weights is None:
            weights = {'completion_time': 0.4, 'workload_balance': 0.3, 'flow_time': 0.2, 'throughput': 0.1}
            
        # Create optimization model
        model = pulp.LpProblem("Zone_Picking_Multi_Objective", pulp.LpMinimize)
        
        # Decision variables
        # x[order][zone][time] = 1 if order is processed in zone at time
        x = {}
        for order in self.orders:
            x[order.order_id] = {}
            for zone in self.zones:
                x[order.order_id][zone.zone_id] = {}
                for t in range(self.time_periods):
                    x[order.order_id][zone.zone_id][t] = pulp.LpVariable(
                        f"x_{order.order_id}_{zone.zone_id}_{t}", cat='Binary'
                    )
        
        # y[order][time] = 1 if order completes at time
        y = {}
        for order in self.orders:
            y[order.order_id] = {}
            for t in range(self.time_periods):
                y[order.order_id][t] = pulp.LpVariable(
                    f"y_{order.order_id}_{t}", cat='Binary'
                )
        
        # Auxiliary variables for objectives
        max_utilization = pulp.LpVariable("max_utilization", lowBound=0)
        min_utilization = pulp.LpVariable("min_utilization", lowBound=0)
        total_flow_time = pulp.LpVariable("total_flow_time", lowBound=0)
        
        # Constraints
        self._add_constraints(model, x, y, max_utilization, min_utilization, total_flow_time)
        
        # Multi-objective function (weighted sum)
        objective = (
            weights['completion_time'] * pulp.lpSum(t * y[order.order_id][t] 
                                                  for order in self.orders for t in range(self.time_periods)) +
            weights['workload_balance'] * (max_utilization - min_utilization) +
            weights['flow_time'] * total_flow_time -
            weights['throughput'] * pulp.lpSum(y[order.order_id][t] 
                                              for order in self.orders for t in range(self.time_periods))
        )
        
        model.setObjective(objective)
        
        # Solve model
        model.solve(pulp.PULP_CBC_CMD(msg=False))
        
        # Extract solution
        return self._extract_solution(model, x, y)
    
    def _add_constraints(self, model, x, y, max_util, min_util, total_flow_time):
        """Add all constraints to the model"""
        
        # Constraint 1: Each order must be processed in each required zone exactly once
        for order in self.orders:
            for zone_id, item_count in order.items_by_zone.items():
                model += pulp.lpSum(x[order.order_id][zone_id][t] for t in range(self.time_periods)) == 1
        
        # Constraint 2: Zone capacity constraints
        for zone in self.zones:
            for t in range(self.time_periods):
                # Calculate total processing time in zone at time t
                total_processing = pulp.lpSum(
                    x[order.order_id][zone.zone_id][t] * 
                    order.items_by_zone.get(zone.zone_id, 0) * zone.pick_time
                    for order in self.orders
                )
                # Capacity: zone.capacity items/hour = zone.capacity/60 items/minute
                model += total_processing <= (zone.capacity / 60) * zone.workers
        
        # Constraint 3: Sequential processing (pick-and-pass)
        for order in self.orders:
            required_zones = sorted(order.items_by_zone.keys())
            for i in range(len(required_zones) - 1):
                current_zone = required_zones[i]
                next_zone = required_zones[i + 1]
                for t in range(self.time_periods - 1):
                    # If order finishes in current zone at t, it can start in next zone at t+handoff_time
                    model += (
                        pulp.lpSum(x[order.order_id][current_zone][tau] for tau in range(t+1)) <=
                        pulp.lpSum(x[order.order_id][next_zone][tau] for tau in range(max(0, t - self.handoff_time + 1), self.time_periods))
                    )
        
        # Constraint 4: Completion time definition
        for order in self.orders:
            required_zones = sorted(order.items_by_zone.keys())
            last_zone = required_zones[-1] if required_zones else 0
            for t in range(self.time_periods):
                # Order completes at t if it's processed in last zone at t
                model += y[order.order_id][t] >= x[order.order_id][last_zone][t] * 0.9
                model += y[order.order_id][t] <= x[order.order_id][last_zone][t] * 1.1
            
            # Each order completes exactly once
            model += pulp.lpSum(y[order.order_id][t] for t in range(self.time_periods)) == 1
        
        # Constraint 5: Zone utilization bounds
        for zone in self.zones:
            utilization = pulp.lpSum(
                x[order.order_id][zone.zone_id][t] * 
                order.items_by_zone.get(zone.zone_id, 0) * zone.pick_time
                for order in self.orders for t in range(self.time_periods)
            ) / (zone.capacity * self.time_periods / 60)
            
            model += max_util >= utilization
            model += min_util <= utilization
        
        # Constraint 6: Flow time calculation
        for order in self.orders:
            completion_time = pulp.lpSum(t * y[order.order_id][t] for t in range(self.time_periods))
            # Simplified flow time (completion time - start time)
            model += total_flow_time >= completion_time
    
    def _extract_solution(self, model, x, y) -> Dict:
        """Extract solution from the solved model"""
        if model.status != pulp.LpStatusOptimal:
            return {'status': 'No optimal solution found'}
        
        # Extract assignments
        assignments = {}
        completion_times = {}
        zone_utilizations = {}
        
        for order in self.orders:
            assignments[order.order_id] = {}
            for zone in self.zones:
                assignments[order.order_id][zone.zone_id] = None
                for t in range(self.time_periods):
                    if pulp.value(x[order.order_id][zone.zone_id][t]) > 0.5:
                        assignments[order.order_id][zone.zone_id] = t
                        break
            
            # Extract completion time
            for t in range(self.time_periods):
                if pulp.value(y[order.order_id][t]) > 0.5:
                    completion_times[order.order_id] = t
                    break
        
        # Calculate zone utilizations
        for zone in self.zones:
            total_processing = 0
            for order in self.orders:
                if zone.zone_id in order.items_by_zone:
                    time_slot = assignments[order.order_id][zone.zone_id]
                    if time_slot is not None:
                        total_processing += order.items_by_zone[zone.zone_id] * zone.pick_time
            
            capacity_time = (zone.capacity / 60) * zone.workers * self.time_periods
            zone_utilizations[zone.zone_id] = total_processing / capacity_time if capacity_time > 0 else 0
        
        # Calculate objectives
        total_completion_time = sum(completion_times.values())
        max_util = max(zone_utilizations.values()) if zone_utilizations else 0
        min_util = min(zone_utilizations.values()) if zone_utilizations else 0
        workload_balance = max_util - min_util
        throughput = len(self.orders) / max(total_completion_time, 1)
        
        return {
            'status': 'Optimal',
            'assignments': assignments,
            'completion_times': completion_times,
            'zone_utilizations': zone_utilizations,
            'objectives': {
                'total_completion_time': total_completion_time,
                'workload_balance': workload_balance,
                'throughput': throughput,
                'total_flow_time': total_completion_time  # Simplified
            }
        }

In [ ]:
# Create and run the optimizer
optimizer = MultiObjectiveZoneOptimizer(zones, orders, handoff_time)
solution = optimizer.solve_multi_objective()

print("Multi-Objective Optimization Results:")
print(f"Status: {solution['status']}")

if solution['status'] == 'Optimal':
    print("\nOrder Assignments:")
    for order_id, assignments in solution['assignments'].items():
        print(f"  Order {order_id}: {assignments}")
    
    print("\nCompletion Times:")
    for order_id, completion_time in solution['completion_times'].items():
        print(f"  Order {order_id}: Time {completion_time}")
    
    print("\nZone Utilizations:")
    for zone_id, utilization in solution['zone_utilizations'].items():
        print(f"  Zone {zone_id}: {utilization:.3f} ({utilization*100:.1f}%)")
    
    print("\nObjective Values:")
    for obj_name, obj_value in solution['objectives'].items():
        print(f"  {obj_name}: {obj_value:.3f}")

In [ ]:
# Visualization of results
def visualize_solution(solution, zones, orders):
    """Create comprehensive visualizations of the optimization solution"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Zone Picking Multi-Objective Optimization Results', fontsize=16, fontweight='bold')
    
    # 1. Zone Utilization Bar Chart
    zone_ids = list(solution['zone_utilizations'].keys())
    utilizations = list(solution['zone_utilizations'].values())
    
    axes[0, 0].bar(zone_ids, utilizations, color='skyblue', alpha=0.7)
    axes[0, 0].set_xlabel('Zone ID')
    axes[0, 0].set_ylabel('Utilization')
    axes[0, 0].set_title('Zone Utilization Levels')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Add capacity reference lines
    for i, zone in enumerate(zones):
        axes[0, 0].axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Target (80%)' if i == 0 else '')
    
    # 2. Order Completion Timeline
    completion_times = list(solution['completion_times'].values())
    order_ids = list(solution['completion_times'].keys())
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(order_ids)))
    axes[0, 1].barh(order_ids, completion_times, color=colors, alpha=0.7)
    axes[0, 1].set_xlabel('Completion Time')
    axes[0, 1].set_ylabel('Order ID')
    axes[0, 1].set_title('Order Completion Timeline')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Workload Balance Scatter Plot
    zone_labels = [f"Zone {zid}" for zid in zone_ids]
    axes[1, 0].scatter(zone_labels, utilizations, s=100, c='orange', alpha=0.7)
    axes[1, 0].set_xlabel('Zone')
    axes[1, 0].set_ylabel('Utilization')
    axes[1, 0].set_title('Workload Balance Analysis')
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Add balance line
    avg_utilization = np.mean(utilizations)
    axes[1, 0].axhline(y=avg_utilization, color='green', linestyle='--', alpha=0.7, label=f'Average: {avg_utilization:.3f}')
    axes[1, 0].legend()
    
    # 4. Objective Values Radar Chart
    objectives = solution['objectives']
    obj_names = list(objectives.keys())
    obj_values = list(objectives.values())
    
    # Normalize values for radar chart
    normalized_values = []
    for i, (name, value) in enumerate(objectives.items()):
        if name == 'workload_balance':
            normalized_values.append(1 - value)  # Lower is better, so invert
        elif name == 'throughput':
            normalized_values.append(min(value / 0.5, 1))  # Normalize to [0,1]
        else:
            normalized_values.append(1 - value / max(obj_values[:3]))  # Normalize others
    
    angles = np.linspace(0, 2 * np.pi, len(obj_names), endpoint=False).tolist()
    normalized_values += normalized_values[:1]  # Complete the circle
    angles += angles[:1]
    
    axes[1, 1] = plt.subplot(2, 2, 4, projection='polar')
    axes[1, 1].plot(angles, normalized_values, 'o-', linewidth=2, color='purple')
    axes[1, 1].fill(angles, normalized_values, alpha=0.25, color='purple')
    axes[1, 1].set_xticks(angles[:-1])
    axes[1, 1].set_xticklabels([name.replace('_', '\n') for name in obj_names])
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].set_title('Objective Performance (Normalized)')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nSolution Summary:")
    print(f"  Total completion time: {solution['objectives']['total_completion_time']:.1f} minutes")
    print(f"  Workload balance variance: {solution['objectives']['workload_balance']:.3f}")
    print(f"  Throughput: {solution['objectives']['throughput']:.3f} orders/minute")
    print(f"  Average zone utilization: {avg_utilization:.3f} ({avg_utilization*100:.1f}%)")

# Visualize the solution
if solution['status'] == 'Optimal':
    visualize_solution(solution, zones, orders)

In [ ]:
# Sensitivity Analysis: Different weight combinations
def sensitivity_analysis():
    """Perform sensitivity analysis with different objective weightings"""
    
    weight_scenarios = [
        {'name': 'Balanced', 'weights': {'completion_time': 0.4, 'workload_balance': 0.3, 'flow_time': 0.2, 'throughput': 0.1}},
        {'name': 'Time_Focused', 'weights': {'completion_time': 0.6, 'workload_balance': 0.2, 'flow_time': 0.1, 'throughput': 0.1}},
        {'name': 'Balance_Focused', 'weights': {'completion_time': 0.2, 'workload_balance': 0.5, 'flow_time': 0.2, 'throughput': 0.1}},
        {'name': 'Throughput_Focused', 'weights': {'completion_time': 0.3, 'workload_balance': 0.2, 'flow_time': 0.1, 'throughput': 0.4}}
    ]
    
    results = []
    
    for scenario in weight_scenarios:
        print(f"\nTesting {scenario['name']} scenario...")
        result = optimizer.solve_multi_objective(scenario['weights'])
        
        if result['status'] == 'Optimal':
            results.append({
                'scenario': scenario['name'],
                'completion_time': result['objectives']['total_completion_time'],
                'workload_balance': result['objectives']['workload_balance'],
                'throughput': result['objectives']['throughput'],
                'avg_utilization': np.mean(list(result['zone_utilizations'].values()))
            })
    
    # Create comparison visualization
    if results:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Sensitivity Analysis: Different Objective Weightings', fontsize=16, fontweight='bold')
        
        scenarios = [r['scenario'] for r in results]
        
        # Completion Time Comparison
        axes[0, 0].bar(scenarios, [r['completion_time'] for r in results], color='lightblue', alpha=0.7)
        axes[0, 0].set_title('Total Completion Time')
        axes[0, 0].set_ylabel('Minutes')
        axes[0, 0].grid(True, alpha=0.3)
        
        # Workload Balance Comparison
        axes[0, 1].bar(scenarios, [r['workload_balance'] for r in results], color='lightgreen', alpha=0.7)
        axes[0, 1].set_title('Workload Balance (Lower is Better)')
        axes[0, 1].set_ylabel('Variance')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Throughput Comparison
        axes[1, 0].bar(scenarios, [r['throughput'] for r in results], color='salmon', alpha=0.7)
        axes[1, 0].set_title('Throughput')
        axes[1, 0].set_ylabel('Orders/Minute')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Average Utilization Comparison
        axes[1, 1].bar(scenarios, [r['avg_utilization'] for r in results], color='gold', alpha=0.7)
        axes[1, 1].set_title('Average Zone Utilization')
        axes[1, 1].set_ylabel('Utilization Ratio')
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print comparison table
        print("\n" + "="*80)
        print("SENSITIVITY ANALYSIS RESULTS")
        print("="*80)
        print(f"{'Scenario':<20} {'Completion':<12} {'Balance':<10} {'Throughput':<12} {'Avg Util':<10}")
        print("-"*70)
        
        for result in results:
            print(f"{result['scenario']:<20} {result['completion_time']:<12.1f} {result['workload_balance']:<10.3f} "
                  f"{result['throughput']:<12.3f} {result['avg_utilization']:<10.3f}")

# Run sensitivity analysis
sensitivity_analysis()

### Why This Tier Exists vs Earlier Tiers

This mathematical formulation tier serves as the foundational baseline for the zone picking problem. It provides:

**Advantages:**
- **Optimal Solutions**: Guarantees mathematically optimal solutions for the given problem instance
- **Multi-Objective Framework**: Simultaneously optimizes competing objectives (time, balance, throughput)
- **Rigorous Analysis**: Provides provable bounds and sensitivity analysis capabilities
- **Benchmark Standard**: Establishes performance baseline for comparing heuristic and advanced algorithms

**Disadvantages:**
- **Computational Complexity**: Exponential time complexity makes it unsuitable for large-scale problems
- **Model Rigidity**: Requires precise mathematical formulation, difficult to adapt to dynamic conditions
- **Solving Time**: Can be slow for medium-sized instances (10+ orders, 5+ zones)
- **Assumption Sensitivity**: Performance depends heavily on model assumptions and parameters

### When to Use This Tier

- **Small to Medium Problems**: Up to 15 orders and 6 zones where optimality is required
- **Strategic Planning**: Long-term facility design and capacity planning decisions
- **Benchmarking**: Establishing performance standards for other algorithms
- **Academic Research**: Understanding the mathematical structure of the problem
- **High-Value Operations**: Where optimal solution justifies computational cost

### Key Insights from Mathematical Formulation

The multi-objective optimization reveals important trade-offs:

1. **Pareto Frontier**: No single solution optimizes all objectives simultaneously
2. **Balance vs Speed**: Better workload balance often increases total completion time
3. **Sequential Constraints**: Pick-and-pass requirements create complex dependency chains
4. **Capacity Bottlenecks**: Zone capacity constraints are the primary limiting factor

This mathematical foundation provides the theoretical basis for understanding the zone picking problem and serves as the benchmark against which all subsequent tiers will be compared.